# 09 — Production Pipeline & SHAP Analysis

**Purpose:** Implement the final Claude few-shot pipeline, evaluate on val data, and provide comprehensive SHAP interpretability analysis.

**Evaluation approach:**

| | Val (167 rows) | January+ articles |
|---|---|---|
| Labelled? | Yes | No |
| Used for | This notebook — F1, SHAP | Inference pipeline (later) |
| Reports | Model performance metrics | Curator alignment rate |
| When | Now | After pipeline is built |

**Sections:**
1. Load models and val data
2. Val set evaluation with Claude few-shot
3. SHAP on TF-IDF (global feature importance)
4. SHAP on Sentence Transformer (best trained model interpretability)
5. Individual prediction explanations
6. Proxy concentration score per class
7. SHAP interaction: where models agree vs disagree
8. Production readiness

# 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import joblib
import shap
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from anthropic import Anthropic
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

%matplotlib inline

shap.initjs()

In [ ]:
DATA_DIR = Path("../data/modelling")
MODEL_DIR = Path("../models")

client = Anthropic()
CLAUDE_MODEL = "claude-sonnet-4-20250514"

# 1. Load models and test data

In [ ]:
# Load trained models
tfidf_vec = joblib.load(MODEL_DIR / "baseline_tfidf.joblib")
tfidf_clf = joblib.load(MODEL_DIR / "baseline_logreg.joblib")
sbert_clf = joblib.load(MODEL_DIR / "sbert_classifier.joblib")
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

with open(DATA_DIR / "metadata_columns.json") as f:
    metadata_cols = json.load(f)

label_names = sorted(pd.read_csv(DATA_DIR / "val.csv")["target"].unique())

# Load val data (for SHAP and threshold analysis)
val_df = pd.read_csv(DATA_DIR / "val.csv")
y_val = val_df["target"]

# Generate val predictions
sbert_emb_val = np.load(MODEL_DIR / "sbert_val_embeddings.npy")
X_val_sbert = np.hstack([sbert_emb_val, val_df[metadata_cols].values])
sbert_proba_val = sbert_clf.predict_proba(X_val_sbert)
sbert_pred_val = sbert_clf.predict(X_val_sbert)
sbert_classes = sbert_clf.classes_

print(f"Val: {len(val_df)} rows")
print(f"Models loaded.")

# Run hybrid on val set
THRESHOLD = 0.40  # from notebook 07 threshold analysis

print(f"Evaluating on: val set ({len(val_df)} articles)")
print(f"Threshold: {THRESHOLD}\n")

hybrid_results = hybrid_classify(val_df, threshold=THRESHOLD)

In [ ]:
# Evaluate hybrid
y_true = val_df["target"]
y_pred = hybrid_results["category_1"]

# Filter valid predictions
valid_mask = y_pred.isin(label_names)

macro_f1 = f1_score(y_true[valid_mask], y_pred[valid_mask], average="macro")
top1_acc = (y_pred[valid_mask] == y_true[valid_mask]).mean()

# Top-2
in_top2 = ((y_pred == y_true) | (hybrid_results["category_2"] == y_true))
top2_acc = in_top2[valid_mask].mean()

print(f"Hybrid — Macro F1: {macro_f1:.3f}")
print(f"Top-1 accuracy: {top1_acc:.3f}")
print(f"Top-2 accuracy: {top2_acc:.3f}")
print(f"\n{classification_report(y_true[valid_mask], y_pred[valid_mask])}")

# Breakdown by source
for source in ["sentence_transformer", "claude"]:
    mask = (hybrid_results["source"] == source) & valid_mask
    if mask.sum() > 0:
        acc = (y_pred[mask] == y_true[mask]).mean()
        print(f"{source}: {mask.sum()} articles, accuracy {acc:.3f}")

In [ ]:
SYSTEM_PROMPT = """You are a classifier for an education policy newsletter. 
Classify each article into exactly one of these categories:

{categories}

Respond with ONLY a JSON object in this format:
{{
  "category": "the_category_name",
  "confidence": 0.0 to 1.0,
  "second_category": "the_second_most_likely_category",
  "second_confidence": 0.0 to 1.0
}}

Do not include any other text."""


def format_system_prompt(category_descriptions, examples=None):
    categories = "\n".join(
        f"- {name}: {desc}" for name, desc in category_descriptions.items()
    )
    prompt = SYSTEM_PROMPT.format(categories=categories)
    if examples:
        prompt += "\n\nHere are example articles for each category:\n" + examples
    return prompt


def classify_with_claude(text, system_prompt):
    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=150,
        system=system_prompt,
        messages=[{"role": "user", "content": f"Classify this article:\n\n{text}"}],
    )
    try:
        return json.loads(response.content[0].text)
    except json.JSONDecodeError:
        return {"category": "parse_error", "confidence": 0.0, "second_category": "parse_error", "second_confidence": 0.0}

In [ ]:
def hybrid_classify(texts_df, threshold=0.40):
    """Hybrid classification: Sentence Transformer for confident, Claude for uncertain."""
    
    # Step 1: Encode all articles with sentence transformer
    embeddings = sbert_model.encode(texts_df["text_clean"].tolist(), show_progress_bar=True)
    
    # Add metadata
    meta_features = texts_df[metadata_cols].reindex(columns=metadata_cols, fill_value=0).values
    X = np.hstack([embeddings, meta_features])
    
    # Step 2: Get ST predictions and confidence
    proba = sbert_clf.predict_proba(X)
    preds = sbert_clf.predict(X)
    max_conf = proba.max(axis=1)
    
    # Step 3: Route low-confidence to Claude
    system_prompt = format_system_prompt(CATEGORY_DESCRIPTIONS, examples=examples_text)
    results = []
    n_claude = 0
    
    for i in range(len(texts_df)):
        if max_conf[i] >= threshold:
            # Sentence Transformer handles it
            top2_idx = np.argsort(proba[i])[-2:][::-1]
            results.append({
                "category_1": sbert_classes[top2_idx[0]],
                "confidence_1": proba[i][top2_idx[0]],
                "category_2": sbert_classes[top2_idx[1]],
                "confidence_2": proba[i][top2_idx[1]],
                "source": "sentence_transformer",
            })
        else:
            # Claude handles it
            claude_result = classify_with_claude(texts_df.iloc[i]["text_clean"], system_prompt)
            results.append({
                "category_1": claude_result.get("category", "error"),
                "confidence_1": claude_result.get("confidence", 0.0),
                "category_2": claude_result.get("second_category", "error"),
                "confidence_2": claude_result.get("second_confidence", 0.0),
                "source": "claude",
            })
            n_claude += 1
            time.sleep(0.5)
    
    print(f"ST handled: {len(texts_df) - n_claude} ({(len(texts_df) - n_claude)/len(texts_df):.0%})")
    print(f"Claude handled: {n_claude} ({n_claude/len(texts_df):.0%})")
    
    return pd.DataFrame(results)

# 3. Test set evaluation

First and only time touching the held-out test data.

In [ ]:
# Run hybrid on test set (or val as placeholder)
THRESHOLD = 0.40  # from notebook 07 threshold analysis

if HAS_TEST:
    eval_df = test_df
    eval_label = "Test set"
else:
    eval_df = val_df
    eval_label = "Val set (placeholder — no test data yet)"

print(f"Evaluating on: {eval_label} ({len(eval_df)} articles)")
print(f"Threshold: {THRESHOLD}\n")

hybrid_results = hybrid_classify(eval_df, threshold=THRESHOLD)

In [ ]:
# Evaluate hybrid
if "target" in eval_df.columns:
    y_true = eval_df["target"]
    y_pred = hybrid_results["category_1"]
    
    # Filter valid predictions
    valid_mask = y_pred.isin(label_names)
    
    macro_f1 = f1_score(y_true[valid_mask], y_pred[valid_mask], average="macro")
    top1_acc = (y_pred[valid_mask] == y_true[valid_mask]).mean()
    
    # Top-2
    in_top2 = ((y_pred == y_true) | (hybrid_results["category_2"] == y_true))
    top2_acc = in_top2[valid_mask].mean()
    
    print(f"Hybrid — Macro F1: {macro_f1:.3f}")
    print(f"Top-1 accuracy: {top1_acc:.3f}")
    print(f"Top-2 accuracy: {top2_acc:.3f}")
    print(f"\n{classification_report(y_true[valid_mask], y_pred[valid_mask])}")
    
    # Breakdown by source
    for source in ["sentence_transformer", "claude"]:
        mask = (hybrid_results["source"] == source) & valid_mask
        if mask.sum() > 0:
            acc = (y_pred[mask] == y_true[mask]).mean()
            print(f"{source}: {mask.sum()} articles, accuracy {acc:.3f}")

# 4. Threshold sensitivity

How much does changing the confidence cutoff change performance? The routing threshold is itself a consequential specification choice.

In [ ]:
# Use val set ST predictions to test different thresholds
# (Don't re-call Claude — use saved few-shot predictions from notebook 06)
claude_few_df = pd.read_csv(DATA_DIR / "claude_few_shot_predictions.csv")
valid_few = claude_few_df[claude_few_df["category"].isin(label_names)]

sbert_max_conf = sbert_proba_val.max(axis=1)

print(f"{'Threshold':<12} {'ST handles':>12} {'Hybrid Top-1':>14} {'Hybrid Top-2':>14}")
print(f"{'-'*52}")

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.60]:
    confident_mask = sbert_max_conf >= threshold
    
    correct_top1 = 0
    correct_top2 = 0
    
    for i in range(len(y_val)):
        true = y_val.iloc[i]
        if confident_mask[i]:
            if sbert_pred_val[i] == true:
                correct_top1 += 1
                correct_top2 += 1
            else:
                top2 = np.argsort(sbert_proba_val[i])[-2:]
                if np.where(sbert_classes == true)[0][0] in top2:
                    correct_top2 += 1
        else:
            if i < len(valid_few):
                claude_row = valid_few.iloc[i]
                if claude_row["category"] == true:
                    correct_top1 += 1
                    correct_top2 += 1
                elif claude_row["second_category"] == true:
                    correct_top2 += 1
    
    pct_st = confident_mask.mean()
    t1 = correct_top1 / len(y_val)
    t2 = correct_top2 / len(y_val)
    print(f"{threshold:<12.2f} {pct_st:>11.0%} {t1:>14.3f} {t2:>14.3f}")

# 5. SHAP on TF-IDF + Logistic Regression

Global feature importance per class — which words drive each category? Compare to TF-IDF coefficients from notebook 03.

In [ ]:
X_val_tfidf = tfidf_vec.transform(val_df["text_clean"])

explainer_tfidf = shap.LinearExplainer(tfidf_clf, X_val_tfidf)
shap_values_tfidf = explainer_tfidf.shap_values(X_val_tfidf)

print(f"SHAP values shape: {len(shap_values_tfidf)} classes, {shap_values_tfidf[0].shape} per class")

In [ ]:
# Per-class top SHAP features
tfidf_feature_names = tfidf_vec.get_feature_names_out()

for i, cls in enumerate(tfidf_clf.classes_):
    mean_shap = np.abs(shap_values_tfidf[i]).mean(axis=0)
    if hasattr(mean_shap, 'A1'):
        mean_shap = mean_shap.A1  # convert sparse matrix to array
    top_idx = np.argsort(mean_shap)[-10:][::-1]
    
    print(f"\n--- {cls} ---")
    for j in top_idx:
        print(f"  {mean_shap[j]:.4f}  {tfidf_feature_names[j]}")

# 6. SHAP on Sentence Transformer + metadata (best model)

Which features drive the best model's decisions? How much do metadata features contribute vs embedding dimensions?

In [ ]:
# KernelExplainer on a sample of 50 articles (slow but thorough)
np.random.seed(42)
sample_idx = np.random.choice(len(X_val_sbert), 50, replace=False)
X_sample = X_val_sbert[sample_idx]

# Background data for KernelExplainer
background = shap.kmeans(X_val_sbert, 10)

print("Running KernelExplainer on 50 articles — this will take several minutes...")
explainer_sbert = shap.KernelExplainer(sbert_clf.predict_proba, background)
shap_values_sbert = explainer_sbert.shap_values(X_sample, nsamples=100)

print(f"Done. SHAP values shape: {len(shap_values_sbert)} classes")

In [ ]:
# Feature names: embedding dimensions + metadata columns
feature_names_sbert = [f"emb_{i}" for i in range(384)] + metadata_cols

# Which features matter most overall?
for i, cls in enumerate(label_names):
    mean_shap = np.abs(shap_values_sbert[i]).mean(axis=0)
    top_idx = np.argsort(mean_shap)[-10:][::-1]
    
    print(f"\n--- {cls} ---")
    n_meta = 0
    for j in top_idx:
        name = feature_names_sbert[j]
        is_meta = "[metadata]" if name in metadata_cols else ""
        if is_meta:
            n_meta += 1
        print(f"  {mean_shap[j]:.4f}  {name} {is_meta}")
    print(f"  → {n_meta}/10 top features are metadata")

# 7. Individual prediction explanations

Waterfall plots for specific interesting cases: disagreements, confident wrong predictions, low-confidence correct predictions.

In [ ]:
# Find interesting cases in the sample
sample_preds = sbert_pred_val[sample_idx]
sample_true = y_val.values[sample_idx]
sample_conf = sbert_proba_val[sample_idx].max(axis=1)

# Case 1: High-confidence wrong prediction
wrong_mask = sample_preds != sample_true
if wrong_mask.any():
    wrong_confs = sample_conf.copy()
    wrong_confs[~wrong_mask] = 0
    case1_idx = np.argmax(wrong_confs)
    print(f"Case 1 — High-confidence wrong prediction:")
    print(f"  True: {sample_true[case1_idx]}, Predicted: {sample_preds[case1_idx]} ({sample_conf[case1_idx]:.2f})")
    print(f"  Text: {val_df.iloc[sample_idx[case1_idx]]['text_clean'][:150]}")

# Case 2: Low-confidence correct prediction
correct_mask = sample_preds == sample_true
if correct_mask.any():
    correct_confs = sample_conf.copy()
    correct_confs[~correct_mask] = 1
    case2_idx = np.argmin(correct_confs)
    print(f"\nCase 2 — Low-confidence correct prediction:")
    print(f"  True: {sample_true[case2_idx]}, Predicted: {sample_preds[case2_idx]} ({sample_conf[case2_idx]:.2f})")
    print(f"  Text: {val_df.iloc[sample_idx[case2_idx]]['text_clean'][:150]}")

In [ ]:
# Waterfall plot for Case 1: high-confidence wrong prediction
if wrong_mask.any():
    pred_class_idx = np.where(np.array(label_names) == sample_preds[case1_idx])[0][0]
    
    explanation = shap.Explanation(
        values=shap_values_sbert[pred_class_idx][case1_idx],
        base_values=explainer_sbert.expected_value[pred_class_idx],
        data=X_sample[case1_idx],
        feature_names=feature_names_sbert,
    )
    
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(explanation, max_display=15, show=False)
    plt.title(f"Case 1: Predicted {sample_preds[case1_idx]} (wrong — actual {sample_true[case1_idx]})")
    plt.tight_layout()
    plt.show()

In [ ]:
# Waterfall plot for Case 2: low-confidence correct prediction
if correct_mask.any():
    pred_class_idx = np.where(np.array(label_names) == sample_preds[case2_idx])[0][0]
    
    explanation = shap.Explanation(
        values=shap_values_sbert[pred_class_idx][case2_idx],
        base_values=explainer_sbert.expected_value[pred_class_idx],
        data=X_sample[case2_idx],
        feature_names=feature_names_sbert,
    )
    
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(explanation, max_display=15, show=False)
    plt.title(f"Case 2: Predicted {sample_preds[case2_idx]} (correct — low confidence {sample_conf[case2_idx]:.2f})")
    plt.tight_layout()
    plt.show()

# 8. Proxy concentration score per class

What percentage of the model's decision-making comes from metadata features (proxies) vs embedding dimensions (semantic content)? This is the quantitative proxy concentration metric for the SST paper.

In [ ]:
# Proxy features = metadata columns (organisation type, item type)
PROXY_INDICES = [i for i, name in enumerate(feature_names_sbert) if name in metadata_cols]
CONTENT_INDICES = [i for i in range(384)]  # embedding dimensions

print(f"{'Category':<45} {'Proxy %':>10} {'Content %':>12} {'Assessment'}")
print(f"{'-'*85}")

proxy_scores = {}
for i, cls in enumerate(label_names):
    total_shap = np.abs(shap_values_sbert[i]).sum(axis=1).mean()
    proxy_shap = np.abs(shap_values_sbert[i][:, PROXY_INDICES]).sum(axis=1).mean()
    content_shap = np.abs(shap_values_sbert[i][:, CONTENT_INDICES]).sum(axis=1).mean()
    
    proxy_pct = proxy_shap / total_shap
    content_pct = content_shap / total_shap
    proxy_scores[cls] = proxy_pct
    
    assessment = "HIGH proxy" if proxy_pct > 0.40 else "moderate" if proxy_pct > 0.20 else "low proxy"
    print(f"{cls:<45} {proxy_pct:>9.1%} {content_pct:>11.1%} {assessment}")

print(f"\nAverage proxy concentration: {np.mean(list(proxy_scores.values())):.1%}")

# 9. SHAP interaction: where models agree vs disagree

For articles where Claude and the Sentence Transformer disagree, what features drove the ST's prediction? Shows the mechanism behind normative divergence.

In [ ]:
# Load Claude zero-shot predictions for comparison
claude_zero_df = pd.read_csv(DATA_DIR / "claude_zero_shot_predictions.csv")
valid_zero = claude_zero_df[claude_zero_df["category"].isin(label_names)]

# Find disagreements in the SHAP sample
claude_preds_sample = valid_zero["category"].values[sample_idx]
st_preds_sample = sample_preds

disagree_in_sample = np.where(claude_preds_sample != st_preds_sample)[0]
print(f"Disagreements in SHAP sample: {len(disagree_in_sample)}/{len(sample_idx)}")

# Show SHAP for disagreement cases
for case_num, idx in enumerate(disagree_in_sample[:3]):  # show first 3
    true_label = sample_true[idx]
    st_pred = st_preds_sample[idx]
    claude_pred = claude_preds_sample[idx]
    
    print(f"\n--- Disagreement {case_num + 1} ---")
    print(f"  True: {true_label}")
    print(f"  ST predicted: {st_pred} ({sample_conf[idx]:.2f})")
    print(f"  Claude predicted: {claude_pred}")
    print(f"  Text: {val_df.iloc[sample_idx[idx]]['text_clean'][:150]}")
    
    # Top SHAP features for ST's prediction
    pred_class_idx = np.where(np.array(label_names) == st_pred)[0][0]
    shap_vals = shap_values_sbert[pred_class_idx][idx]
    top_features = np.argsort(np.abs(shap_vals))[-5:][::-1]
    print(f"  ST driven by:")
    for j in top_features:
        print(f"    {shap_vals[j]:+.4f}  {feature_names_sbert[j]}")

# 10. Production readiness

In [ ]:
import time as timer

# Latency test: single article through ST
test_text = val_df.iloc[0]["text_clean"]

start = timer.time()
emb = sbert_model.encode([test_text])
meta = val_df.iloc[[0]][metadata_cols].reindex(columns=metadata_cols, fill_value=0).values
X = np.hstack([emb, meta])
pred = sbert_clf.predict(X)
st_latency = timer.time() - start

print(f"Sentence Transformer latency: {st_latency*1000:.0f}ms")
print(f"\nProduction estimates (15 articles/week):")
print(f"  All ST:     {15 * st_latency:.1f}s total, $0/week")
print(f"  All Claude: ~15-30s total, ~$0.15/week (~$8/year)")
print(f"  Hybrid ({int(THRESHOLD*100)}% threshold): ~{15*0.6*st_latency + 15*0.4*1.5:.0f}s, ~$0.06/week (~$3/year)")
print(f"\nEdge cases to handle:")
print(f"  - New category appears: update prompt + retrain ST")
print(f"  - New organisation: add to metadata mapping in s02_preprocess.py")
print(f"  - Claude API down: fall back to ST-only (91.6% top-2 still usable)")

# Summary & Conclusions

*To be completed after running the notebook.*

**Next:** Build inference pipeline (step 7 in decisions doc) — connect the hybrid model to `src/pipeline.py`.